In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Overlap-Untangled 3-Class Disaster Triage: Clinical Composite Indices + Anomaly Proximity + Noise Cleaning + Random Forest (`models/train_distant_analysis.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and executes the **Recommended Multi-Stage Disentanglement Pipeline** for Random Forest on heavily overlapping clinical emergency vitals across the 3 disaster triage tiers:

```mermaid
flowchart TD
    Raw["Raw 8 Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median')"]
    Imputer --> Eng["1. Clinical Composite Feature Engineering (+4 Indices: SI, PP, ROX, Age-SI)"]
    Eng --> Anomaly["2. Unsupervised Anomaly Proximity (+1 Feature: IsolationForest score)"]
    Anomaly --> Clean["3. Borderline Noise Cleaning (NeighbourhoodCleaningRule on Train)"]
    Clean --> RF["4. Random Forest Classifier (300 Trees, class_weight=None on R^13 Space)"]
    RF --> Eval["5. Full Holdout Test Set Evaluation (83k Visits) & Clinical Decision Plots"]
```

### 🎯 3-Tier Disaster Triage Acuity Mapping
1. **`Tier 0: RED (ESI 1)`** ($y=0$): Immediate Resuscitation / Life Threat ($5,271$ visits, $\sim 0.94\%$).
2. **`Tier 1: YELLOW (ESI 2–3)`** ($y=1$): Emergent & Urgent conditions ($440,059$ visits, $\sim 78.86\%$).
3. **`Tier 2: GREEN (ESI 4–5)`** ($y=2$): Semi-urgent & Non-urgent conditions ($112,699$ visits, $\sim 20.19\%$).

### 🩺 13 Feature Space Breakdown
- **Core Arrival Vitals & Demographics (8)**:
  `age`, `cc_breathingdifficulty`, `gender`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_dbp`, `triage_vital_rr`, `triage_vital_o2`
- **Clinical Composite Indices (4 - Disentangles Diagonal Overlap)**:
  1. `shock_index = HR / SBP` (Hemodynamic / Cardiovascular shock indicator)
  2. `pulse_pressure = SBP - DBP` (Vascular tone / Stroke volume proxy)
  3. `rox_index = SpO2 / RR` (Hypoxic respiratory failure index)
  4. `age_shock_index = Age * Shock_Index` (Age-adjusted cardiovascular collapse)
- **Unsupervised Abnormality Proximity (1 - Isolates Severe Resuscitation Outliers)**:
  5. `iso_anomaly_score` (Continuous isolation anomaly score from `IsolationForest`)

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data from R, Partition & Median Imputation
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from imblearn.under_sampling import NeighbourhoodCleaningRule, RandomUnderSampler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

CORE_FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# 3-Tier Categorical Acuity Labels:
# 0: RED (ESI 1), 1: YELLOW (ESI 2-3), 2: GREEN (ESI 4-5)
y_3tier_all = np.zeros(len(esi_all), dtype=np.int32)
y_3tier_all[esi_all == 1] = 0                          # Tier 0: RED (ESI 1)
y_3tier_all[np.isin(esi_all, [2, 3])] = 1              # Tier 1: YELLOW (ESI 2-3)
y_3tier_all[np.isin(esi_all, [4, 5])] = 2              # Tier 2: GREEN (ESI 4-5)
TIER_LABELS = ['RED (ESI 1)', 'YELLOW (ESI 2-3)', 'GREEN (ESI 4-5)']

print("=========================================================================")
print("     5v_cleandf 3-TIER DISASTER TRIAGE COHORT (CLINICAL PIPELINE)")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(esi_all):,}")
print(f"  * Tier 0 [RED (ESI 1)]     : {np.sum(y_3tier_all == 0):,} ({np.mean(y_3tier_all == 0)*100:.2f}%)")
print(f"  * Tier 1 [YELLOW (ESI 2-3)]: {np.sum(y_3tier_all == 1):,} ({np.mean(y_3tier_all == 1)*100:.2f}%)")
print(f"  * Tier 2 [GREEN (ESI 4-5)] : {np.sum(y_3tier_all == 2):,} ({np.mean(y_3tier_all == 2)*100:.2f}%)")
print(f"Core Features ({len(CORE_FEATURES)}): {CORE_FEATURES}")
print("=========================================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_3tier_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_3tier_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_3tier_all[itr]
y_val   = y_3tier_all[iva]
y_test  = y_3tier_all[ite]

# Fit SimpleImputer strictly on Training partition
print("Fitting SimpleImputer(strategy='median') on Training set...")
imputer = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_tr)
X_val_imp = imputer.transform(raw_val)
X_te_imp  = imputer.transform(raw_te)

print(f"✓ Processed Imputed Shapes: Train={X_tr_imp.shape}, Val={X_val_imp.shape}, Test={X_te_imp.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Clinical Composite Features Engineering & Isolation Forest Anomaly Score
# ---------------------------------------------------------------------------
def compute_clinical_composite_features(X_mat, feat_list=CORE_FEATURES):
    """
    Engineers 4 non-linear clinical composite indices to untangle diagonal vital overlap:
      1. shock_index      = HR / SBP
      2. pulse_pressure   = SBP - DBP
      3. rox_index        = SpO2 / RR
      4. age_shock_index  = Age * Shock_Index
    """
    age_idx = feat_list.index('age')
    hr_idx  = feat_list.index('triage_vital_hr')
    sbp_idx = feat_list.index('triage_vital_sbp')
    dbp_idx = feat_list.index('triage_vital_dbp')
    rr_idx  = feat_list.index('triage_vital_rr')
    o2_idx  = feat_list.index('triage_vital_o2')
    
    hr_vec  = X_mat[:, hr_idx]
    sbp_vec = np.clip(X_mat[:, sbp_idx], 30.0, 300.0)
    dbp_vec = X_mat[:, dbp_idx]
    rr_vec  = np.clip(X_mat[:, rr_idx], 4.0, 60.0)
    o2_vec  = np.clip(X_mat[:, o2_idx], 50.0, 100.0)
    age_vec = X_mat[:, age_idx]
    
    shock_index    = hr_vec / sbp_vec
    pulse_pressure = sbp_vec - dbp_vec
    rox_index      = o2_vec / rr_vec
    age_shock_idx  = age_vec * shock_index
    
    return np.column_stack([X_mat, shock_index, pulse_pressure, rox_index, age_shock_idx])


print("Engineering 4 Composite Clinical Indices across Train, Val, and Test splits...")
X_tr_comp  = compute_clinical_composite_features(X_tr_imp)
X_val_comp = compute_clinical_composite_features(X_val_imp)
X_te_comp  = compute_clinical_composite_features(X_te_imp)

print(f"✓ Composite Features Matrix Shape: {X_tr_comp.shape} (12 Total Features)")

# Step 4: Fit Unsupervised Isolation Forest to produce Continuous Abnormality Score
print("\nFitting IsolationForest(contamination=0.01) on Training Set for Anomaly Proximity Feature...")
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_tr_comp)

# Append anomaly scores (more negative = more severe physiological deviation)
train_anomaly = iso_forest.score_samples(X_tr_comp)
val_anomaly   = iso_forest.score_samples(X_val_comp)
test_anomaly  = iso_forest.score_samples(X_te_comp)

X_train_full = np.column_stack([X_tr_comp, train_anomaly])
X_val_full   = np.column_stack([X_val_comp, val_anomaly])
X_test_full  = np.column_stack([X_te_comp, test_anomaly])

ALL_13_FEATURES = CORE_FEATURES + [
    'shock_index', 'pulse_pressure', 'rox_index', 'age_shock_index', 'iso_anomaly_score'
]

print(f"✓ Final Disentangled Feature Space: {X_train_full.shape[1]} Features")
print(f"  Features ({len(ALL_13_FEATURES)}): {ALL_13_FEATURES}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Borderline Noise Cleaning on Training Partition (Neighbourhood Cleaning Rule)
# ---------------------------------------------------------------------------
print("Applying NeighbourhoodCleaningRule on Training Set to remove ambiguous boundary noise...")

# NCR cleans overlapping majority samples whose nearest neighbors belong to opposite classes
ncr = NeighbourhoodCleaningRule(
    n_neighbors=3,
    threshold_cleaning=0.5,
    n_jobs=-1
)

# Subsample majority if needed for instant speed, or fit directly
X_train_clean, y_train_clean = ncr.fit_resample(X_train_full, y_train)

print(f"✓ Cleaned Training Cohort Shape: {X_train_clean.shape}")
print(f"  * RED (ESI 1)     : {np.sum(y_train_clean == 0):,} visits (Preserved 100%)")
print(f"  * YELLOW (ESI 2-3) : {np.sum(y_train_clean == 1):,} visits (Cleaned {np.sum(y_train == 1) - np.sum(y_train_clean == 1):,} boundary noise samples)")
print(f"  * GREEN (ESI 4-5)  : {np.sum(y_train_clean == 2):,} visits (Cleaned {np.sum(y_train == 2) - np.sum(y_train_clean == 2):,} boundary noise samples)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Train Random Forest Classifier (No Class Weighting on Disentangled Space)
# ---------------------------------------------------------------------------
print("Training Random Forest on Disentangled 13-Feature Cleaned Training Space...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight=None,  # No class weighting per user request
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_clean, y_train_clean)

# Check performance on unaltered validation set
pred_val = rf_model.predict(X_val_full)
val_acc  = accuracy_score(y_val, pred_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)

print(f"✓ Random Forest Successfully Trained!")
print(f"  * Number of Trees   : {rf_model.n_estimators}")
print(f"  * Tree Max Depth    : {rf_model.max_depth}")
print(f"  * Class Weighting   : None")
print(f"  * Validation Accuracy: {val_acc*100:.2f}% | Balanced Accuracy: {val_bacc*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Holdout Test Set Evaluation (83,705 Unaltered Visits)
# ---------------------------------------------------------------------------
pred_test = rf_model.predict(X_test_full)
p_test    = rf_model.predict_proba(X_test_full)

acc_3tier     = accuracy_score(y_test, pred_test)
bal_acc_3tier = balanced_accuracy_score(y_test, pred_test)
rec_per_class = recall_score(y_test, pred_test, average=None)
prec_per_class= precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per_class  = f1_score(y_test, pred_test, average=None, zero_division=0)
macro_f1      = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1   = f1_score(y_test, pred_test, average='weighted', zero_division=0)

report_rows = [
    {
        'Triage_Tier': 'Tier 0: RED (ESI 1)',
        'True_Visits': int(np.sum(y_test == 0)),
        'Predicted_Visits': int(np.sum(pred_test == 0)),
        'Sensitivity (Recall)': round(rec_per_class[0], 4),
        'Precision': round(prec_per_class[0], 4),
        'F1_Score': round(f1_per_class[0], 4)
    },
    {
        'Triage_Tier': 'Tier 1: YELLOW (ESI 2-3)',
        'True_Visits': int(np.sum(y_test == 1)),
        'Predicted_Visits': int(np.sum(pred_test == 1)),
        'Sensitivity (Recall)': round(rec_per_class[1], 4),
        'Precision': round(prec_per_class[1], 4),
        'F1_Score': round(f1_per_class[1], 4)
    },
    {
        'Triage_Tier': 'Tier 2: GREEN (ESI 4-5)',
        'True_Visits': int(np.sum(y_test == 2)),
        'Predicted_Visits': int(np.sum(pred_test == 2)),
        'Sensitivity (Recall)': round(rec_per_class[2], 4),
        'Precision': round(prec_per_class[2], 4),
        'F1_Score': round(f1_per_class[2], 4)
    }
]

report_df = pd.DataFrame(report_rows)
print("=====================================================================================================================")
print("  HOLDOUT TEST EVALUATION: DISENTANGLED CLINICAL PREPROCESSING + RANDOM FOREST (3-TIER)")
print("=====================================================================================================================")
print(f"Total Test Cohort Evaluated: {len(y_test):,} visits")
print(f"Overall Accuracy         : {acc_3tier*100:.2f}%")
print(f"Macro Balanced Accuracy  : {bal_acc_3tier*100:.2f}%")
print(f"Macro F1-Score           : {macro_f1:.4f}")
print(f"Weighted F1-Score        : {weighted_f1:.4f}\n")
print(report_df.to_string(index=False))
print("=====================================================================================================================\n")

print("Detailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'clinical_rf_3class_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: 3-Class Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=TIER_LABELS,
    yticklabels=TIER_LABELS
)

ax.set_title(
    f'Disentangled Clinical RF: 3-Class Confusion Matrix (Holdout Test)\n'
    f'Accuracy: {acc_3tier*100:.2f}% | Macro Balanced Accuracy: {bal_acc_3tier*100:.2f}%',
    fontsize=11.5,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Predicted Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('True Disaster Triage Tier', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'distant_analysis', 'clinical_rf_3class_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'clinical_rf_3class_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Feature Importance Bar Chart (13 Features)
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': ALL_13_FEATURES,
    'Importance': importances,
    'Feature_Type': [
        'Core Arrival Vital' if f in CORE_FEATURES else ('Engineered Composite' if f != 'iso_anomaly_score' else 'Anomaly Proximity')
        for f in ALL_13_FEATURES
    ]
}).sort_values(by='Importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6.5))
palette = {'Core Arrival Vital': '#1f77b4', 'Engineered Composite': '#ff7f0e', 'Anomaly Proximity': '#d62728'}
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', hue='Feature_Type', palette=palette, dodge=False, ax=ax)

ax.set_title('Random Forest Feature Importances across 13 Clinical & Engineered Features', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Normalized Gini Importance', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(loc='lower right', fontsize=10.5, title='Feature Category')

plt.tight_layout()
imp_file = os.path.join(plots_dir, 'distant_analysis', 'clinical_rf_feature_importance.png')
plt.savefig(imp_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'clinical_rf_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Feature importance plot saved to: {imp_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: 2D Clinical Decision Boundaries over Engineered Composite Planes
# ---------------------------------------------------------------------------
medians_test = np.median(X_test_full, axis=0)

planes = [
    ('shock_index', 'pulse_pressure', 'Shock Index (HR / SBP)', 'Pulse Pressure (SBP - DBP)', (0.3, 2.0), (10, 140), 'Hemodynamic Disentanglement Plane (SI vs PP)'),
    ('triage_vital_hr', 'triage_vital_sbp', 'Heart Rate (bpm)', 'Systolic BP (mmHg)', (30, 200), (50, 240), 'Raw Hemodynamic Plane (HR vs SBP)'),
    ('rox_index', 'iso_anomaly_score', 'ROX Index (SpO2 / RR)', 'Isolation Anomaly Score', (1.5, 16.0), (-0.7, -0.2), 'Respiratory vs Global Anomaly Plane'),
    ('age', 'age_shock_index', 'Age (years)', 'Age-Shock Index (Age * SI)', (18, 95), (10, 150), 'Cardiovascular Aging Plane')
]

fig, axes = plt.subplots(2, 2, figsize=(17, 14))
axes = axes.flatten()

np.random.seed(42)
test_red_idx    = np.where(y_test == 0)[0]
test_yellow_idx = np.where(y_test == 1)[0]
test_green_idx  = np.where(y_test == 2)[0]
test_yellow_sample = np.random.choice(test_yellow_idx, min(1200, len(test_yellow_idx)), replace=False)
test_green_sample  = np.random.choice(test_green_idx, min(1200, len(test_green_idx)), replace=False)

for idx, (f1_name, f2_name, x_lbl, y_lbl, x_lim, y_lim, plane_title) in enumerate(planes):
    f1_idx = ALL_13_FEATURES.index(f1_name)
    f2_idx = ALL_13_FEATURES.index(f2_name)
    
    xx, yy = np.meshgrid(np.linspace(x_lim[0], x_lim[1], 150), np.linspace(y_lim[0], y_lim[1], 150))
    grid_points = np.tile(medians_test, (xx.size, 1))
    grid_points[:, f1_idx] = xx.ravel()
    grid_points[:, f2_idx] = yy.ravel()
    
    preds_grid = rf_model.predict(grid_points).reshape(xx.shape)
    
    ax = axes[idx]
    ax.contourf(xx, yy, preds_grid, levels=[-0.5, 0.5, 1.5, 2.5], colors=['#ffcccc', '#ffe6cc', '#ccffcc'], alpha=0.65)
    ax.contour(xx, yy, preds_grid, levels=[0.5, 1.5], colors='black', linewidths=1.6, linestyles='--')
    
    ax.scatter(X_test_full[test_green_sample, f1_idx], X_test_full[test_green_sample, f2_idx],
               c='#2ca02c', alpha=0.40, s=16, label='GREEN (ESI 4-5)')
    ax.scatter(X_test_full[test_yellow_sample, f1_idx], X_test_full[test_yellow_sample, f2_idx],
               c='#ff7f0e', alpha=0.35, s=16, label='YELLOW (ESI 2-3)')
    ax.scatter(X_test_full[test_red_idx, f1_idx], X_test_full[test_red_idx, f2_idx],
               c='#d62728', alpha=0.85, s=34, edgecolors='black', linewidth=0.5, label='RED (ESI 1)')
    
    ax.set_title(f'Random Forest: {plane_title}', fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel(x_lbl, fontsize=10.5, fontweight='bold')
    ax.set_ylabel(y_lbl, fontsize=10.5, fontweight='bold')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.grid(True, linestyle=':', alpha=0.4)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)

plt.suptitle('3-Class Clinical Random Forest Decision Boundaries over Composite Planes', fontsize=14.5, fontweight='bold', y=0.995)
plt.tight_layout()

db_file = os.path.join(plots_dir, 'distant_analysis', 'clinical_rf_decision_boundaries.png')
plt.savefig(db_file, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'clinical_rf_decision_boundaries.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Decision boundaries saved to: {db_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 10: Export Production Bundle & Deployment Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

clinical_rf_bundle = {
    'imputer': imputer,
    'compute_composite_fn': compute_clinical_composite_features,
    'iso_forest': iso_forest,
    'rf_model': rf_model,
    'core_features': CORE_FEATURES,
    'all_features': ALL_13_FEATURES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'clinical_rf_3class_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(clinical_rf_bundle, f)

manifest = dict(
    pipeline_architecture='Clinical_Composite_Indices_Plus_Anomaly_Plus_Random_Forest',
    preprocessing=[
        'Median_Imputation',
        'Composite_Clinical_Feature_Engineering_SI_PP_ROX_AgeSI',
        'Unsupervised_Isolation_Forest_Anomaly_Score',
        'Neighbourhood_Cleaning_Rule_Boundary_Denoising'
    ],
    classifier='RandomForestClassifier',
    n_estimators=int(rf_model.n_estimators),
    max_depth=int(rf_model.max_depth),
    class_weight=None,
    dataset='5v_cleandf_RData',
    core_features=CORE_FEATURES,
    all_features=ALL_13_FEATURES,
    n_features=len(ALL_13_FEATURES),
    tier_labels=TIER_LABELS,
    total_samples=len(esi_all),
    n_red=int(np.sum(y_3tier_all == 0)),
    n_yellow=int(np.sum(y_3tier_all == 1)),
    n_green=int(np.sum(y_3tier_all == 2)),
    overall_accuracy=round(acc_3tier, 4),
    macro_balanced_accuracy=round(bal_acc_3tier, 4),
    macro_f1=round(macro_f1, 4),
    holdout_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'clinical_rf_3class_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported Production Bundle  : {bundle_file}")
print(f"✓ Exported Production Manifest: {manifest_file}")